## ⚡ Delta Lake Optimization & Maintenance

**Purpose**: Optimize tables for query performance
- **OPTIMIZE**: Compact small files for better read performance
- **Z-ORDER**: Co-locate related data for faster filtering
- **VACUUM**: Remove old file versions (after retention period)

In [0]:
%python
# Optimize Delta tables for query performance
print("=== Delta Lake Table Optimization ===")

# Optimize key tables
print("\nOptimizing silver_transactions...")
spark.sql(f"OPTIMIZE {CATALOG}.{SCHEMA}.silver_transactions")
print("✓ Optimized")

print("\nOptimizing silver_customers with Z-ORDER on customer_id...")
spark.sql(f"OPTIMIZE {CATALOG}.{SCHEMA}.silver_customers ZORDER BY (customer_id)")
print("✓ Optimized with Z-ORDER")

print("\nOptimizing gold_customer_insights...")
spark.sql(f"OPTIMIZE {CATALOG}.{SCHEMA}.gold_customer_insights")
print("✓ Optimized")

print("\n✓ Table optimization complete - improved query performance")
print("\nNote: VACUUM can be run to remove old file versions:")
print("  Example: VACUUM silver_transactions RETAIN 168 HOURS")

In [0]:
-- Sample analytical queries for business users

-- Top 5 customers by revenue
SELECT 
  customer_name,
  city,
  total_revenue,
  total_transactions,
  completion_rate
FROM workspace.dataflow_platform.gold_customer_insights
ORDER BY total_revenue DESC
LIMIT 5;

In [0]:
-- Product performance by category
SELECT 
  category,
  COUNT(*) as product_count,
  SUM(total_revenue) as category_revenue,
  AVG(success_rate) as avg_success_rate,
  SUM(unique_customers) as total_customers
FROM workspace.dataflow_platform.gold_product_performance
GROUP BY category
ORDER BY category_revenue DESC;

In [0]:
-- Daily revenue trend with 7-day moving average
SELECT 
  transaction_date,
  daily_revenue,
  revenue_7day_avg,
  transaction_count,
  unique_customers,
  ROUND((daily_revenue - revenue_7day_avg) / revenue_7day_avg * 100, 2) as pct_vs_avg
FROM workspace.dataflow_platform.gold_daily_sales
ORDER BY transaction_date DESC
LIMIT 10;

## ⏱️ Delta Lake Features - Time Travel & Version Control

**Purpose**: Leverage Delta Lake's ACID transactions and versioning
- Query historical data (time travel)
- View table history and versions
- Rollback to previous versions
- Track schema evolution

In [0]:
%python
# View Delta table history
print("=== Delta Lake Version History ===")
print("\nSilver Customers Table History:")
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCHEMA}.silver_customers") \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(10, truncate=False)

print("\nGold Customer Insights Table History:")
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCHEMA}.gold_customer_insights") \
    .select('version', 'timestamp', 'operation') \
    .show(5, truncate=False)

In [0]:
%python
# Time Travel - Query previous version before SCD merge
print("=== Time Travel Example ===")
print("\nCustomers at Version 0 (before SCD updates):")
df_version_0 = spark.read.format('delta') \
    .option('versionAsOf', 0) \
    .table(f'{CATALOG}.{SCHEMA}.silver_customers')

print(f"Total records at version 0: {df_version_0.count()}")
df_version_0.filter(col('customer_id').isin([1, 2])) \
    .select('customer_id', 'customer_name', 'city', 'customer_status', 'is_current') \
    .show()

print("\nCustomers at current version (after SCD updates):")
df_current = spark.table(f'{CATALOG}.{SCHEMA}.silver_customers')
print(f"Total records at current version: {df_current.count()}")
df_current.filter(col('customer_id').isin([1, 2])) \
    .select('customer_id', 'customer_name', 'city', 'customer_status', 'is_current') \
    .orderBy('customer_id', 'effective_start_date') \
    .show()

print("✓ Time travel demonstrates version tracking")

In [0]:
%python
# Schema change detection
print("=== Schema Evolution Tracking ===")

# Function to compare schemas
def detect_schema_changes(table_name):
    history = spark.sql(f"DESCRIBE HISTORY {table_name}").collect()
    
    print(f"\nTable: {table_name}")
    print(f"Total versions: {len(history)}")
    
    # Check for schema changes
    schema_changes = [h for h in history if 'schema' in str(h['operationParameters']).lower()]
    
    if schema_changes:
        print(f"Schema changes detected: {len(schema_changes)} times")
    else:
        print("No explicit schema changes")
    
    # Get current schema
    current_schema = spark.table(table_name).schema
    print(f"Current columns: {len(current_schema.fields)}")
    print(f"Column names: {[f.name for f in current_schema.fields]}")
    
    return len(current_schema.fields)

# Check schema for key tables
detect_schema_changes(f'{CATALOG}.{SCHEMA}.silver_customers')
detect_schema_changes(f'{CATALOG}.{SCHEMA}.gold_customer_insights')

print("\n✓ Schema tracking enabled via Delta Lake versioning")

In [0]:
%python
# Final Data Quality and Pipeline Summary
print("="*70)
print("    DATAFLOW INC. - DATA PLATFORM SUMMARY REPORT")
print("="*70)

def get_table_stats(table_name):
    df = spark.table(f'{CATALOG}.{SCHEMA}.{table_name}')
    return df.count(), len(df.columns)

# Bronze Layer
print("\n🥉 BRONZE LAYER (Raw Ingestion)")
for table in ['bronze_transactions', 'bronze_customers', 'bronze_products', 'bronze_logs']:
    rows, cols = get_table_stats(table)
    print(f"  ✓ {table:25s} | {rows:6d} rows | {cols:2d} columns")

# Silver Layer
print("\n🥈 SILVER LAYER (Cleaned & Validated)")
for table in ['silver_transactions', 'silver_customers', 'silver_products', 'silver_logs']:
    rows, cols = get_table_stats(table)
    print(f"  ✓ {table:25s} | {rows:6d} rows | {cols:2d} columns")

# Gold Layer
print("\n🥇 GOLD LAYER (Business Metrics)")
for table in ['gold_customer_insights', 'gold_product_performance', 'gold_daily_sales', 'gold_event_analytics']:
    rows, cols = get_table_stats(table)
    print(f"  ✓ {table:25s} | {rows:6d} rows | {cols:2d} columns")

print("\n" + "="*70)
print("CAPABILITIES IMPLEMENTED:")
print("  ✓ Medallion Architecture (Bronze → Silver → Gold)")
print("  ✓ Data Quality Checks (nulls, duplicates, business rules)")
print("  ✓ SCD Type 2 (Historical tracking with effective dates)")
print("  ✓ Delta Lake ACID transactions & versioning")
print("  ✓ Time Travel (query previous versions)")
print("  ✓ Schema Evolution tracking")
print("  ✓ Window Functions (trend analysis)")
print("  ✓ Complex JOINs (customer-product-transaction)")
print("  ✓ Aggregations & KPIs for analytics")
print("="*70)

print("\n✨ Data Platform Ready for Production Analytics!")

## 📊 SCD Type 2 - Slowly Changing Dimensions

**Purpose**: Track historical changes to customer data over time
- Maintain full audit trail of changes
- Use effective dates to track validity periods
- Support point-in-time queries
- Implement using MERGE statement

**Columns**:
- `effective_start_date`: When this version became active
- `effective_end_date`: When this version expired (NULL = current)
- `is_current`: Boolean flag for current record

In [0]:
%python
# Simulate incoming customer updates (status changes and relocations)
import pandas as pd
from datetime import datetime

# Create updated customer records with changes
updated_customers_data = [
    {'customer_id': 1, 'customer_name': 'Customer_1', 'email': 'customer1@example.com', 
     'city': 'Seattle', 'signup_date': '2025-01-15', 'customer_status': 'active'},  # Changed city
    {'customer_id': 2, 'customer_name': 'Customer_2', 'email': 'customer2@example.com',
     'city': 'Los Angeles', 'signup_date': '2025-02-10', 'customer_status': 'inactive'},  # Changed status
    {'customer_id': 21, 'customer_name': 'Customer_21', 'email': 'customer21@example.com',
     'city': 'Boston', 'signup_date': '2026-07-01', 'customer_status': 'active'}  # New customer
]

# Create staging DataFrame
df_customer_updates = spark.createDataFrame(updated_customers_data) \
    .withColumn('signup_date', to_date(col('signup_date'))) \
    .withColumn('email', lower(col('email'))) \
    .withColumn('customer_status', upper(col('customer_status')))

print("✓ Customer updates prepared:")
print(f"  - Updates: {df_customer_updates.count()} records")
df_customer_updates.show()

In [0]:
%python
# Implement SCD Type 2 using MERGE statement
from delta.tables import DeltaTable

# Read existing customers table
target_table = DeltaTable.forName(spark, f'{CATALOG}.{SCHEMA}.silver_customers')

# Prepare staging data with SCD columns
df_staging = df_customer_updates \
    .withColumn('effective_start_date', current_date()) \
    .withColumn('effective_end_date', lit(None).cast('date')) \
    .withColumn('is_current', lit(True)) \
    .withColumn('silver_timestamp', current_timestamp())

# Step 1: Expire old records (set effective_end_date and is_current = False)
target_table.alias('target') \
    .merge(
        df_staging.alias('source'),
        "target.customer_id = source.customer_id AND target.is_current = true"
    ) \
    .whenMatchedUpdate(
        condition="""target.city != source.city 
                      OR target.customer_status != source.customer_status""",
        set={
            'effective_end_date': current_date(),
            'is_current': lit(False)
        }
    ) \
    .execute()

print("✓ Step 1: Old records expired")

# Step 2: Insert new versions (both changed records and new customers)
# For changed records, insert new version; for new customers, insert as is
df_existing_customers = spark.table(f'{CATALOG}.{SCHEMA}.silver_customers')

# Find which customers have changes or are new
df_changes = df_staging.alias('src') \
    .join(
        df_existing_customers.filter(col('is_current') == False).alias('tgt'),
        'customer_id',
        'left'
    ) \
    .where(
        (col('tgt.customer_id').isNotNull()) |  # Changed customers (now have expired records)
        (col('tgt.customer_id').isNull())        # New customers
    ) \
    .select('src.*')

# Insert new versions
df_changes.write.format('delta') \
    .mode('append') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_customers')

print("✓ Step 2: New versions inserted")
print(f"\n✓ SCD Type 2 merge complete")

In [0]:
%python
# Verify SCD Type 2 implementation
df_customers_all = spark.table(f'{CATALOG}.{SCHEMA}.silver_customers')

print("=== All Customer Records (Current + Historical) ===")
print(f"Total records: {df_customers_all.count()}")

# Show records with history for customer_id = 1 and 2
print("\nCustomer 1 history (city changed from original to Seattle):")
df_customers_all.filter(col('customer_id') == 1) \
    .select('customer_id', 'customer_name', 'city', 'customer_status', 
            'effective_start_date', 'effective_end_date', 'is_current') \
    .orderBy('effective_start_date') \
    .show(truncate=False)

print("Customer 2 history (status changed):")
df_customers_all.filter(col('customer_id') == 2) \
    .select('customer_id', 'customer_name', 'city', 'customer_status',
            'effective_start_date', 'effective_end_date', 'is_current') \
    .orderBy('effective_start_date') \
    .show(truncate=False)

print("\n=== Current Records Only ===")
df_customers_all.filter(col('is_current') == True) \
    .select('customer_id', 'customer_name', 'city', 'customer_status', 'is_current') \
    .orderBy('customer_id') \
    .show(5)

print("✓ SCD Type 2 verified - Historical tracking working correctly")

## 🥇 Gold Layer - Business-Ready Analytics

**Purpose**: Aggregated metrics for business intelligence and analytics
- Pre-aggregated KPIs
- Denormalized fact tables
- Business-specific metrics
- Optimized for BI tools and dashboards

In [0]:
%python
# Join transactions with customers and products for customer insights
df_trans = spark.table(f'{CATALOG}.{SCHEMA}.silver_transactions')
df_cust = spark.table(f'{CATALOG}.{SCHEMA}.silver_customers')

# Customer transaction summary
df_customer_insights = df_trans \
    .join(df_cust, 'customer_id', 'inner') \
    .groupBy('customer_id', 'customer_name', 'city', 'customer_status') \
    .agg(
        count('transaction_id').alias('total_transactions'),
        sum(when(col('status') == 'COMPLETED', 1).otherwise(0)).alias('completed_transactions'),
        sum('amount').alias('total_revenue'),
        avg('amount').alias('avg_transaction_amount'),
        max('transaction_date').alias('last_transaction_date'),
        min('transaction_date').alias('first_transaction_date')
    ) \
    .withColumn('completion_rate', 
                round(col('completed_transactions') / col('total_transactions') * 100, 2)) \
    .withColumn('gold_timestamp', current_timestamp())

df_customer_insights.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_customer_insights')

print(f"✓ Customer insights created: {df_customer_insights.count()} rows")
df_customer_insights.orderBy(col('total_revenue').desc()).show(5)

In [0]:
%python
# Product performance metrics
df_prod = spark.table(f'{CATALOG}.{SCHEMA}.silver_products')

df_product_performance = df_trans \
    .join(df_prod, 'product_id', 'inner') \
    .groupBy('product_id', 'product_name', 'category', 'price') \
    .agg(
        count('transaction_id').alias('total_sales'),
        sum('amount').alias('total_revenue'),
        countDistinct('customer_id').alias('unique_customers'),
        sum(when(col('status') == 'COMPLETED', 1).otherwise(0)).alias('successful_sales')
    ) \
    .withColumn('avg_revenue_per_sale', round(col('total_revenue') / col('total_sales'), 2)) \
    .withColumn('success_rate', round(col('successful_sales') / col('total_sales') * 100, 2)) \
    .withColumn('gold_timestamp', current_timestamp())

df_product_performance.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_product_performance')

print(f"✓ Product performance created: {df_product_performance.count()} rows")
df_product_performance.orderBy(col('total_revenue').desc()).show(5)

In [0]:
%python
# Daily sales summary with trend analysis (window functions)
df_daily_sales = df_trans \
    .filter(col('status') == 'COMPLETED') \
    .groupBy('transaction_date') \
    .agg(
        count('transaction_id').alias('transaction_count'),
        sum('amount').alias('daily_revenue'),
        avg('amount').alias('avg_order_value'),
        countDistinct('customer_id').alias('unique_customers')
    )

# Add window functions for trend analysis
window_spec = Window.orderBy('transaction_date').rowsBetween(-6, 0)

df_daily_sales_trend = df_daily_sales \
    .withColumn('revenue_7day_avg', avg('daily_revenue').over(window_spec)) \
    .withColumn('revenue_7day_sum', sum('daily_revenue').over(window_spec)) \
    .withColumn('day_rank_by_revenue', 
                dense_rank().over(Window.orderBy(col('daily_revenue').desc()))) \
    .withColumn('gold_timestamp', current_timestamp()) \
    .orderBy('transaction_date')

df_daily_sales_trend.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_daily_sales')

print(f"✓ Daily sales trend created: {df_daily_sales_trend.count()} rows")
df_daily_sales_trend.orderBy(col('transaction_date').desc()).show(5)

In [0]:
%python
# Event analytics from logs
df_logs = spark.table(f'{CATALOG}.{SCHEMA}.silver_logs')

df_event_analytics = df_logs \
    .groupBy('event_date', 'event_type') \
    .agg(
        count('log_id').alias('event_count'),
        countDistinct('user_id').alias('unique_users'),
        avg('session_duration').alias('avg_session_duration')
    ) \
    .withColumn('gold_timestamp', current_timestamp())

df_event_analytics.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_event_analytics')

print(f"✓ Event analytics created: {df_event_analytics.count()} rows")
print(f"\n✓ Gold layer complete - 4 aggregated tables")

## 🥈 Silver Layer - Cleaned & Validated Data

**Purpose**: Clean, validate, and standardize data
- Data quality checks (nulls, duplicates, invalid values)
- Type casting and standardization
- Business rule validation
- Deduplication
- Schema enforcement

In [0]:
%python
# Data Quality Check Function
def check_data_quality(df, table_name, checks):
    """
    Run data quality checks on a DataFrame
    checks: dict of {check_name: check_condition}
    """
    print(f"\n=== Data Quality Report: {table_name} ===")
    results = []
    
    for check_name, condition in checks.items():
        failed_count = df.filter(~condition).count()
        total_count = df.count()
        passed = failed_count == 0
        
        results.append({
            'table': table_name,
            'check': check_name,
            'failed_rows': failed_count,
            'total_rows': total_count,
            'status': '✓ PASS' if passed else '✗ FAIL'
        })
        
        print(f"  {check_name}: {results[-1]['status']} (Failed: {failed_count}/{total_count})")
    
    return results

print("✓ Data quality framework loaded")

In [0]:
%python
# Read from Bronze
df_bronze_trans = spark.table(f'{CATALOG}.{SCHEMA}.bronze_transactions')

# Data Quality Checks
quality_checks = {
    'transaction_id_not_null': col('transaction_id').isNotNull(),
    'customer_id_not_null': col('customer_id').isNotNull(),
    'amount_positive': col('amount') > 0,
    'valid_status': col('status').isin(['completed', 'pending', 'failed'])
}

check_data_quality(df_bronze_trans, 'transactions', quality_checks)

# Clean and transform
df_silver_trans = df_bronze_trans \
    .filter(col('transaction_id').isNotNull()) \
    .filter(col('amount') > 0) \
    .withColumn('transaction_date', to_date(col('transaction_date'))) \
    .withColumn('amount', round(col('amount'), 2)) \
    .withColumn('status', upper(col('status'))) \
    .withColumn('year_month', date_format(col('transaction_date'), 'yyyy-MM')) \
    .withColumn('silver_timestamp', current_timestamp()) \
    .dropDuplicates(['transaction_id'])

# Write to Silver
df_silver_trans.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_transactions')

print(f"\n✓ Silver transactions created: {df_silver_trans.count()} rows")
df_silver_trans.show(5)

In [0]:
%python
# Read from Bronze
df_bronze_cust = spark.table(f'{CATALOG}.{SCHEMA}.bronze_customers')

# Quality Checks
quality_checks = {
    'customer_id_not_null': col('customer_id').isNotNull(),
    'email_valid': col('email').contains('@'),
    'customer_name_not_null': col('customer_name').isNotNull()
}

check_data_quality(df_bronze_cust, 'customers', quality_checks)

# Clean and add SCD Type 2 columns
df_silver_cust = df_bronze_cust \
    .filter(col('customer_id').isNotNull()) \
    .filter(col('email').contains('@')) \
    .withColumn('signup_date', to_date(col('signup_date'))) \
    .withColumn('email', lower(col('email'))) \
    .withColumn('customer_status', upper(col('customer_status'))) \
    .withColumn('silver_timestamp', current_timestamp()) \
    .withColumn('effective_start_date', current_date()) \
    .withColumn('effective_end_date', lit(None).cast('date')) \
    .withColumn('is_current', lit(True)) \
    .dropDuplicates(['customer_id'])

# Write to Silver
df_silver_cust.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_customers')

print(f"\n✓ Silver customers created with SCD Type 2: {df_silver_cust.count()} rows")
df_silver_cust.select('customer_id', 'customer_name', 'city', 'effective_start_date', 'is_current').show(5)

In [0]:
%python
# Products
df_bronze_prod = spark.table(f'{CATALOG}.{SCHEMA}.bronze_products')

df_silver_prod = df_bronze_prod \
    .filter(col('product_id').isNotNull()) \
    .filter(col('price') > 0) \
    .withColumn('price', round(col('price'), 2)) \
    .withColumn('category', initcap(col('category'))) \
    .withColumn('silver_timestamp', current_timestamp()) \
    .dropDuplicates(['product_id'])

df_silver_prod.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_products')

print(f"✓ Silver products: {df_silver_prod.count()} rows")

# Logs
df_bronze_logs = spark.table(f'{CATALOG}.{SCHEMA}.bronze_logs')

df_silver_logs = df_bronze_logs \
    .filter(col('log_id').isNotNull()) \
    .withColumn('timestamp', to_timestamp(col('timestamp'))) \
    .withColumn('event_date', to_date(col('timestamp'))) \
    .withColumn('event_hour', hour(col('timestamp'))) \
    .withColumn('silver_timestamp', current_timestamp()) \
    .dropDuplicates(['log_id'])

df_silver_logs.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_logs')

print(f"✓ Silver logs: {df_silver_logs.count()} rows")
print(f"\n✓ Silver layer complete - 4 cleaned tables")

## 🥉 Bronze Layer - Raw Data Ingestion

**Purpose**: Ingest raw data AS-IS with minimal transformations
- Preserve original data structure
- Add metadata columns (ingestion timestamp, source file)
- Store in Delta format for versioning and time travel

In [0]:
%python
# Use raw transactions DataFrame (created in setup cell)

# Add metadata columns
df_transactions_bronze = df_transactions_raw \
    .withColumn('ingestion_timestamp', current_timestamp()) \
    .withColumn('source_file', lit('simulated_transactions.csv')) \
    .withColumn('bronze_id', monotonically_increasing_id())

# Write to Bronze Delta table
df_transactions_bronze.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_transactions')

print("✓ Transactions loaded to bronze_transactions")
print(f"  Row count: {df_transactions_bronze.count()}")
df_transactions_bronze.show(5)

In [0]:
%python
# Use raw customers DataFrame (created in setup cell)

df_customers_bronze = df_customers_raw \
    .withColumn('ingestion_timestamp', current_timestamp()) \
    .withColumn('source_file', lit('simulated_customers.csv')) \
    .withColumn('bronze_id', monotonically_increasing_id())

df_customers_bronze.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_customers')

print("✓ Customers loaded to bronze_customers")
print(f"  Row count: {df_customers_bronze.count()}")
df_customers_bronze.show(5)

In [0]:
%python
# Use raw products DataFrame (created in setup cell)

df_products_bronze = df_products_raw \
    .withColumn('ingestion_timestamp', current_timestamp()) \
    .withColumn('source_file', lit('simulated_products.json')) \
    .withColumn('bronze_id', monotonically_increasing_id())

df_products_bronze.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_products')

print("✓ Products loaded to bronze_products")

# Use raw logs DataFrame (created in setup cell)

df_logs_bronze = df_logs_raw \
    .withColumn('ingestion_timestamp', current_timestamp()) \
    .withColumn('source_file', lit('simulated_logs.json')) \
    .withColumn('bronze_id', monotonically_increasing_id())

df_logs_bronze.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_logs')

print("✓ Logs loaded to bronze_logs")
print(f"\n✓ Bronze layer complete - 4 tables created")

# DataFlow Inc. - Centralized Data Platform

## Problem Statement
Building a centralized platform to:
- Ingest raw CSV/JSON data (transactions, logs, products, customers)
- Clean and validate data with quality checks
- Detect schema changes and data drift
- Maintain historical data (SCD Type 2)
- Enable reliable analytics

## Architecture: Medallion (Bronze → Silver → Gold)
- **Bronze**: Raw data ingestion (no transformations)
- **Silver**: Cleaned, validated, deduplicated data
- **Gold**: Business-ready aggregated metrics

## Tech Stack
- **Spark + Delta Lake**: Distributed processing with ACID transactions and versioning
- **Python/PySpark**: Data processing
- **SCD Type 2**: Historical tracking with effective dates

In [0]:
%python
# Setup: Create sample data files to simulate raw CSV/JSON sources
import pandas as pd
import json
from datetime import datetime, timedelta
import random
import builtins

# Save Python's built-in round function since it will be overridden by PySpark's round
python_round = builtins.round

# Create sample transactions data (CSV)
transactions_data = []
for i in range(1, 101):
    transactions_data.append({
        'transaction_id': i,
        'customer_id': random.randint(1, 20),
        'product_id': random.randint(1, 10),
        'amount': python_round(random.uniform(10.0, 500.0), 2),
        'transaction_date': (datetime.now() - timedelta(days=random.randint(0, 90))).strftime('%Y-%m-%d'),
        'status': random.choice(['completed', 'pending', 'failed'])
    })

transactions_df = pd.DataFrame(transactions_data)
# transactions_df will be converted to Spark DataFrame below

# Create sample customer data (CSV)
customers_data = []
for i in range(1, 21):
    customers_data.append({
        'customer_id': i,
        'customer_name': f'Customer_{i}',
        'email': f'customer{i}@example.com',
        'city': random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']),
        'signup_date': (datetime.now() - timedelta(days=random.randint(100, 365))).strftime('%Y-%m-%d'),
        'customer_status': random.choice(['active', 'inactive'])
    })

customers_df = pd.DataFrame(customers_data)
# customers_df will be converted to Spark DataFrame below

# Create sample product data (JSON)
products_data = []
for i in range(1, 11):
    products_data.append({
        'product_id': i,
        'product_name': f'Product_{i}',
        'category': random.choice(['Electronics', 'Clothing', 'Home', 'Sports']),
        'price': python_round(random.uniform(20.0, 1000.0), 2),
        'stock_quantity': random.randint(0, 100)
    })

# products_data will be converted to Spark DataFrame below

# Create sample logs data (JSON)
logs_data = []
for i in range(1, 51):
    logs_data.append({
        'log_id': i,
        'timestamp': (datetime.now() - timedelta(hours=random.randint(0, 48))).isoformat(),
        'event_type': random.choice(['login', 'logout', 'purchase', 'view', 'error']),
        'user_id': random.randint(1, 20),
        'ip_address': f'192.168.{random.randint(1, 255)}.{random.randint(1, 255)}',
        'session_duration': random.randint(30, 3600)
    })

# logs_data will be converted to Spark DataFrame below

# Convert to Spark DataFrames directly
global df_transactions_raw, df_customers_raw, df_products_raw, df_logs_raw

df_transactions_raw = spark.createDataFrame(transactions_df)
df_customers_raw = spark.createDataFrame(customers_df)
df_products_raw = spark.createDataFrame(pd.DataFrame(products_data))
df_logs_raw = spark.createDataFrame(pd.DataFrame(logs_data))

print("✓ Sample data created as Spark DataFrames:")
print(f"  - transactions: {df_transactions_raw.count()} rows")
print(f"  - customers: {df_customers_raw.count()} rows")
print(f"  - products: {df_products_raw.count()} records")
print(f"  - logs: {df_logs_raw.count()} records")

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

# Define catalog structure (using workspace catalog)
CATALOG = "workspace"
SCHEMA = "dataflow_platform"

# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"✓ Using catalog: {CATALOG}.{SCHEMA}")
print(f"✓ Spark version: {spark.version}")